In [0]:
%sql
SHOW TABLES IN vivek.practise1;

In [0]:
from pyspark.sql.functions import *

# ===================================
# Create Customer Data
# ===================================

spark.createDataFrame([
    (1, "Vivek", "Chennai"),
    (2, "Rahul", "Bangalore"),
    (3, "Priya", "Mumbai"),
    (4, "Arun", "Hyderabad"),
    (5, "Sneha", "Pune")
]) \
.toDF("id", "name", "city") \
.write.mode("overwrite") \
.format("delta") \
.save("/Volumes/vivek/practise/vkalivasan/customers")


# ===================================
# Create Product Data
# ===================================

spark.createDataFrame([
    (1, "Mobile", 10000),
    (2, "Laptop", 20000),
    (3, "Tab", 5000),
    (4, "Mobile", 8000),
    (5, "Laptop", 240000)
]) \
.toDF("id", "product", "amount") \
.write.mode("overwrite") \
.format("delta") \
.save("/Volumes/vivek/practise/vkalivasan/product")


# ===================================
# Read Delta Files
# ===================================

customer_df = spark.read.format("delta") \
    .load("/Volumes/vivek/practise/vkalivasan/customers")

product_df = spark.read.format("delta") \
    .load("/Volumes/vivek/practise/vkalivasan/product")


print("Customer Data")
customer_df.show()

print("Product Data")
product_df.show()


# ===================================
# Narrow Transformations
# ===================================

# Filter
customer_df.filter(col("city") == "Chennai").show()

# Select
product_df.select("product", "amount").show()

# Filter Amount > 10000
product_df.filter(col("amount") > 10000).show()

# Filter Laptop Products
product_df.filter(col("product") == "Laptop").show()

# WithColumn (2% value)
product_df = product_df.withColumn(
    "discount_amount",
    col("amount") * 2 / 100
)

product_df.show()


# ===================================
# Wide Transformation - Join
# ===================================

final_df = customer_df.join(
    product_df,
    "id",
    "inner"
)

final_df.show()


# ===================================
# Wide Transformation - GroupBy
# ===================================

final_df.groupBy("product") \
    .agg(
        sum("discount_amount").alias("total_discount")
    ) \
    .show()


# ===================================
# Multiple Aggregations
# ===================================

final_df.groupBy("product") \
    .agg(
        count("*").alias("total_orders"),
        sum("amount").alias("total_sales"),
        avg("amount").alias("avg_sales"),
        max("amount").alias("max_sales"),
        min("amount").alias("min_sales")
    ) \
    .show()

final_df.explain()

In [0]:
#cache and persist
df.cache() --> store dataframe in cache
df.persist() --> store dataframe in memory

#Broadcast - No big shuffle

from pyspark.sql.functions import broadcast

joined_df = orders_df.join(
    broadcast(products_df),
    "product_id"
)


df.repartition(8)

df.coalesce(2)

✅ Partitions
✅ Repartition vs Coalesce
✅ Narrow vs Wide
✅ Shuffle
✅ Cache vs Persist
✅ Broadcast Join - Always broadcast the SMALL table, not the large table.
⬜ Explain Plan
⬜ Data Skew
⬜ AQE


#When a Spark job is slow, the first thing experienced engineers do is:

df.explain(True)

#Data skew -Data skew happens when one partition contains much more data than others.

orders_df.join(
    customers_df.hint("broadcast"),
    "customer_id"
)

df.repartition(100)


spark.read.csv()
spark.read.parquet()
spark.table()
spark.sql()
spark.createDataFrame()   # for testing
spark.range()
spark.conf.get()


In [0]:

from pyspark.sql.functions import *
df = spark.range(100).withColumn("date", current_date()).withColumn("name", lit("vivek"))
df.show(20)
spark.conf.get("spark.sql.shuffle.partitions")